In [5]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pymysql
import os
from dotenv import load_dotenv

load_dotenv()

# MySQL Connection
def get_db_connection():
    return pymysql.connect(
        host=os.getenv('MYSQL_HOST'),
        port=int(os.getenv('MYSQL_PORT', 3306)),
        user=os.getenv('MYSQL_USER'),
        password=os.getenv('MYSQL_PASSWORD'),
        database=os.getenv('MYSQL_DATABASE')
    )

In [6]:
conn = get_db_connection()

# Read wisata data from MySQL
query = '''
SELECT 
    kw.nama_kategori as kategori_wisata_id,
    w.nama_tempat,
    w.jam_buka,
    w.jam_tutup,
    w.alamat,
    w.lokasi_geo,
    w.htm_min_domestik,
    w.htm_max_domestik,
    w.htm_min_mancanegara,
    w.htm_max_mancanegara,
    w.akses_transportasi,
    w.rating
FROM wisata w
JOIN kategori_wisata kw ON w.kategori_wisata_id = kw.kategori_wisata_id
'''

df = pd.read_sql(query, conn)
conn.close()

df.head()

C:\Users\mallexibra\AppData\Local\Temp\ipykernel_3408\3500743921.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,kategori_wisata_id,nama_tempat,jam_buka,jam_tutup,alamat,lokasi_geo,htm_min_domestik,htm_max_domestik,htm_min_mancanegara,htm_max_mancanegara,akses_transportasi,rating
0,Wisata Alam,Waduk Tirto Agro -Glenmore,07:30:00,16:00:00,"Dusun Karanganyar, Kajarharjo, Kec. Kalibaru","-8.2788264,113.9324687",10000.0,10000.0,10000.0,10000.0,"Mobil, Sepeda Motor",0.0
1,Wisata Alam,Bukit Mondoleko,05:00:00,17:00:00,"Area Pesawahan, Temuguruh, Kec. Sempu","-8.235249255000987, 114.16452966653851",10000.0,10000.0,10000.0,10000.0,"Mobil, Sepeda Motor",4.0
2,Wisata Alam,Puncak Asmoro,00:00:00,23:59:00,"Kacangan Asri, Gombeng, Gombengsari, Kec. Kali...","-8.13619,114.3332131",10000.0,0.0,10000.0,0.0,Sepeda Motor,4.2
3,Wisata Alam,Bukit Sewu Sambang,00:00:00,23:59:00,"Lingk.Papring, Wangkal, Kalipuro, Kec. Kalipuro","-8.124987190275466, 114.37148215304485",5000.0,15000.0,5000.0,15000.0,"Mobil, Sepeda Motor",4.5
4,Wisata Alam,Air Terjun Temcor,07:00:00,17:00:00,"Sumber Asih, Sumberarum, Kec. Songgon","-8.20023620715303, 114.16053242660568",5000.0,5000.0,5000.0,5000.0,"Mobil, Sepeda Motor",4.5


In [7]:
# Bersihkan dan normalisasi nama kolom
df.columns = df.columns.str.replace('_', ' ').str.title()

# Mapping kolom CSV ke nama yang diharapkan
column_mapping = {
    'Kategori Wisata Id': 'Kategori',
    'Nama Tempat': 'Nama Tempat',
    'Jam Buka': 'Jam Buka',
    'Jam Tutup': 'Jam Tutup',
    'Lokasi Geo': 'GPS Location',
    'Htm Min Domestik': 'HTM Min Domestic',
    'Htm Max Domestik': 'HTM Max Domestic',
    'Htm Min Mancanegara': 'HTM Min Mancanegara',
    'Htm Max Mancanegara': 'HTM Max Mancanegara',
    'Akses Transportasi': 'Moda Transportasi',
    'Rating': 'Rating'
}

df = df.rename(columns=column_mapping)

# Pastikan rating adalah numeric
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce').fillna(0)

df.head()

,Kategori,Nama Tempat,Jam Buka,Jam Tutup,Alamat,GPS Location,HTM Min Domestic,HTM Max Domestic,HTM Min Mancanegara,HTM Max Mancanegara,Moda Transportasi,Rating
0,Wisata Alam,Waduk Tirto Agro -Glenmore,07:30:00,16:00:00,"Dusun Karanganyar, Kajarharjo, Kec. Kalibaru","-8.2788264,113.9324687",10000.0,10000.0,10000.0,10000.0,"Mobil, Sepeda Motor",0.0
1,Wisata Alam,Bukit Mondoleko,05:00:00,17:00:00,"Area Pesawahan, Temuguruh, Kec. Sempu","-8.235249255000987, 114.16452966653851",10000.0,10000.0,10000.0,10000.0,"Mobil, Sepeda Motor",4.0
2,Wisata Alam,Puncak Asmoro,00:00:00,23:59:00,"Kacangan Asri, Gombeng, Gombengsari, Kec. Kali...","-8.13619,114.3332131",10000.0,0.0,10000.0,0.0,Sepeda Motor,4.2
3,Wisata Alam,Bukit Sewu Sambang,00:00:00,23:59:00,"Lingk.Papring, Wangkal, Kalipuro, Kec. Kalipuro","-8.124987190275466, 114.37148215304485",5000.0,15000.0,5000.0,15000.0,"Mobil, Sepeda Motor",4.5
4,Wisata Alam,Air Terjun Temcor,07:00:00,17:00:00,"Sumber Asih, Sumberarum, Kec. Songgon","-8.20023620715303, 114.16053242660568",5000.0,5000.0,5000.0,5000.0,"Mobil, Sepeda Motor",4.5


In [8]:
# Read ratings from MySQL
conn = get_db_connection()

query = '''
SELECT 
    user_id,
    w.nama_tempat as wisata,
    rw.nilai_rating as rating
FROM rating_wisata rw
JOIN wisata w ON rw.wisata_id = w.wisata_id
ORDER BY user_id
'''

ratings = pd.read_sql(query, conn)

# Fallback: generate mock data if no ratings in DB
if len(ratings) == 0:
    ratings = pd.DataFrame({
        'user_id': np.random.randint(1, 15, 80),
        'wisata': np.random.choice(df['Nama Tempat'], 80),
        'rating': np.random.randint(3, 6, 80)
    })

conn.close()
ratings.head()

C:\Users\mallexibra\AppData\Local\Temp\ipykernel_3408\3994941538.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ratings = pd.read_sql(query, conn)


,user_id,wisata,rating
0,11,capas,4
1,1,Bangsring Underwater,5
2,5,Gumuk Kancil Glenmore,5
3,13,Pantai Watudodol,4
4,8,Perkebunan Kaliklatak,4


In [9]:
# Read favorit from MySQL
conn = get_db_connection()

query = '''
SELECT 
    user_id,
    w.nama_tempat as wisata,
    1 as saved
FROM favorit_wisata fw
JOIN wisata w ON fw.wisata_id = w.wisata_id
ORDER BY user_id
'''

saved = pd.read_sql(query, conn)

# Fallback: generate mock data if no favorit in DB
if len(saved) == 0:
    saved = pd.DataFrame({
        'user_id': np.random.randint(1, 15, 40),
        'wisata': np.random.choice(df['Nama Tempat'], 40),
        'saved': 1
    })

conn.close()
saved.head()

C:\Users\mallexibra\AppData\Local\Temp\ipykernel_3408\2278331075.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  saved = pd.read_sql(query, conn)


,user_id,wisata,saved
0,9,Waduk Tirto Agro -Glenmore,1
1,14,Pelangi Sari,1
2,2,Gua Maria Curahjati,1
3,4,Pantai Lampon,1
4,4,Pantai Plengkung,1


In [10]:
interaction = pd.merge(
    ratings,
    saved,
    on=['user_id','wisata'],
    how='outer'
).fillna(0)

interaction['score'] = interaction['rating'] + interaction['saved']

interaction.head()

,user_id,wisata,rating,saved,score
0,1,Air Terjun Legomoro,4.0,0.0,4.0
1,1,Air Terjun Telunjuk Raung,5.0,0.0,5.0
2,1,Bangsring Underwater,5.0,0.0,5.0
3,1,Pantai Grajagan,5.0,0.0,5.0
4,1,Pantai Muncar,4.0,0.0,4.0


In [11]:
user_item_matrix = interaction.pivot_table(
    index='user_id',
    columns='wisata',
    values='rating'
).fillna(0)

user_item_matrix.head()

wisata,Agro Wisata Gunug Gumitir,Air Terjun Jagir,Air Terjun Kalibendo,Air Terjun Kedung Angin,Air Terjun Legomoro,Air Terjun Pertemon,Air Terjun Seloresi,Air Terjun Telunjuk Raung,Air Terjun Temcor,Bangsring Underwater,...,Wong Osing,capas,kali sawah adventure,pantai Trianggulasi,pantai gumuk kantong,pantai pulau santen,sendang seruni,wisata Banyu Kuwung,wisata Beji antaboga,wisata japuro
user_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,4.0,0.0,0.0,5.0,0.0,5.0,...,0.0,0.0,0.0,0.0,0.0,4.0,0.0,3.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,...,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
similarity = cosine_similarity(user_item_matrix)

similarity_df = pd.DataFrame(
    similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

similarity_df.head(10)

user_id,1,2,3,4,5,6,7,8,9,10,11,12,13,14
user_id,,,,,,,,,,,,,,
1,1.000000,0.0,0.150513,0.167956,0.000000,0.000000,0.220854,0.000000,0.108536,0.197371,0.000000,0.318902,0.118423,0.000000
2,0.000000,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.214821,0.000000,0.000000,0.000000
3,0.150513,0.0,1.000000,0.193135,0.000000,0.000000,0.000000,0.144115,0.166410,0.000000,0.000000,0.203728,0.136176,0.000000
4,0.167956,0.0,0.193135,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.202610,0.000000,0.227338,0.151958,0.000000
5,0.000000,0.0,0.000000,0.000000,1.000000,0.337364,0.000000,0.000000,0.000000,0.224281,0.000000,0.000000,0.000000,0.000000
6,0.000000,0.0,0.000000,0.000000,0.337364,1.000000,0.000000,0.000000,0.000000,0.161165,0.000000,0.000000,0.000000,0.000000
7,0.220854,0.0,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.202704,0.000000,0.000000,0.000000
8,0.000000,0.0,0.144115,0.000000,0.000000,0.000000,0.000000,1.000000,0.230940,0.000000,0.000000,0.000000,0.000000,0.305888
9,0.108536,0.0,0.166410,0.000000,0.000000,0.000000,0.000000,0.230940,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [13]:
def get_cf_wisata(user_id):

    similar_users = similarity_df[user_id]\
        .sort_values(ascending=False)[1:6]

    similar_users_index = similar_users.index

    rekomendasi = interaction[
        interaction['user_id'].isin(similar_users_index)
    ]

    rekomendasi = rekomendasi.sort_values(
        by='rating',
        ascending=False
    )

    return rekomendasi['wisata'].unique()

In [14]:
def content_based_filter(
    wisata_list,
    kategori,
    budget_min,
    budget_max,
    rating_min
):

    filtered = df[
        (df['Nama Tempat'].isin(wisata_list)) &
        (df['Kategori'].isin(kategori)) &
        (df['HTM Min Domestic'] >= budget_min) &
        (df['HTM Max Domestic'] <= budget_max) &
        (df['Rating'] >= rating_min)
    ]

    return filtered

In [15]:
def hybrid_recommendation(
    user_id,
    kategori,
    budget_min,
    budget_max,
    rating_min
):

    cf_result = get_cf_wisata(user_id)

    # # fallback jika CF sedikit
    if len(cf_result) < 5:
        cf_result = df['Nama Tempat']

    final = content_based_filter(
        cf_result,
        kategori,
        budget_min,
        budget_max,
        rating_min
    )

    final = final.sort_values(
        by='Rating',
        ascending=False
    )

    return final

In [16]:
hasil = hybrid_recommendation(
    user_id=1,
    kategori=['Wisata Alam', 'Wisata Religi', 'Wisata Edukasi'],
    budget_min=0,
    budget_max=20000,
    rating_min=4
)

print(hasil[['Nama Tempat','Kategori', 'HTM Min Domestic', 'HTM Max Domestic', 'HTM Min Mancanegara', 'HTM Max Mancanegara', 'Rating']])

                   Nama Tempat        Kategori  HTM Min Domestic  \
29                 Teluk Hijau     Wisata Alam            7500.0   
96       Pura Agung Blambangan   Wisata Religi               0.0   
86  Penangkaran Penyu Sukomade  Wisata Edukasi            5000.0   
81          Kebun Kopi Kemiren  Wisata Edukasi               0.0   
9                 Gunung Ranti     Wisata Alam            7500.0   
48        Bangsring Underwater     Wisata Alam            5000.0   
99       Gumuk Kancil Glenmore   Wisata Religi               0.0   
37                Pantai Bedul     Wisata Alam           10000.0   
3           Bukit Sewu Sambang     Wisata Alam            5000.0   
35               Pantai Lampon     Wisata Alam               0.0   
57         wisata Banyu Kuwung     Wisata Alam            5000.0   
38            Pantai Plengkung     Wisata Alam           15000.0   
33                 Pulau Merah     Wisata Alam           10000.0   
98       Makam Datuk Abdurahim   Wisata Religi  

In [17]:
cf_result = get_cf_wisata(2)
cf_result

<StringArray>
[  'Air Terjun Telunjuk Raung',             'Pantai Grajagan',
        'Bangsring Underwater',                 'Pulau Bedil',
 'Penangkaran Banteng Baluran',       'Gumuk Kancil Glenmore',
              'Pantai Cacalan',                'Gunung Raung',
                  'Wong Osing',                'Gunung Ranti',
       'Pura Agung Blambangan',       'Makam Datuk Abdurahim',
        'wisata Beji antaboga',                      'capas ',
      'Pantai Cemara - Kawang',         'pantai pulau santen',
         'Air Terjun Legomoro',               'Pantai Muncar',
                'Pelangi Sari',                'Pantai Bedul',
            'Pantai Rajegwesi',             'Bukit Mondoleko',
         'wisata Banyu Kuwung',             'Pantai Sukamade',
               'TN Alas Purwo',              'Pantai So Long',
        'Pinus reborn (Suwis)',        'kali sawah adventure',
          'Kebun Kopi Kemiren',  'Penangkaran Penyu Sukomade',
         'Air Terjun Seloresi',          

In [18]:
# Test simple import
print("Testing...")
import sys
print(f"Python version: {sys.version}")

Testing...
Python version: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
